In [3]:
import pandas as pd

path = "../data/"

orders = pd.read_csv(path + "olist_orders_dataset.csv")
customers = pd.read_csv(path + "olist_customers_dataset.csv")
items = pd.read_csv(path + "olist_order_items_dataset.csv")
payments = pd.read_csv(path + "olist_order_payments_dataset.csv")
reviews = pd.read_csv(path + "olist_order_reviews_dataset.csv")
products = pd.read_csv(path + "olist_products_dataset.csv")
sellers = pd.read_csv(path + "olist_sellers_dataset.csv")
category_translation = pd.read_csv(path + "product_category_name_translation.csv")

In [4]:
date_cols = [
    "order_purchase_timestamp", "order_approved_at",
    "order_delivered_carrier_date", "order_delivered_customer_date",
    "order_estimated_delivery_date",
]
for col in date_cols:
    orders[col] = pd.to_datetime(orders[col])

orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  object        
 1   customer_id                    99441 non-null  object        
 2   order_status                   99441 non-null  object        
 3   order_purchase_timestamp       99441 non-null  datetime64[ns]
 4   order_approved_at              99281 non-null  datetime64[ns]
 5   order_delivered_carrier_date   97658 non-null  datetime64[ns]
 6   order_delivered_customer_date  96476 non-null  datetime64[ns]
 7   order_estimated_delivery_date  99441 non-null  datetime64[ns]
dtypes: datetime64[ns](5), object(3)
memory usage: 6.1+ MB


In [5]:
# Which statuses have no customer delivery date?
print(orders[orders["order_delivered_customer_date"].isnull()]["order_status"].value_counts())

# Delivered orders with no delivery date
orders[(orders["order_status"] == "delivered") &
       (orders["order_delivered_customer_date"].isnull())]

order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: count, dtype: int64


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,ec05a6d8558c6455f0cbbd8a420ad34f,delivered,2017-11-28 17:44:07,2017-11-28 17:56:40,2017-11-30 18:12:23,NaT,2017-12-18
20618,f5dd62b788049ad9fc0526e3ad11a097,5e89028e024b381dc84a13a3570decb4,delivered,2018-06-20 06:58:43,2018-06-20 07:19:05,2018-06-25 08:05:00,NaT,2018-07-16
43834,2ebdfc4f15f23b91474edf87475f108e,29f0540231702fda0cfdee0a310f11aa,delivered,2018-07-01 17:05:11,2018-07-01 17:15:12,2018-07-03 13:57:00,NaT,2018-07-30
79263,e69f75a717d64fc5ecdfae42b2e8e086,cfda40ca8dd0a5d486a9635b611b398a,delivered,2018-07-01 22:05:55,2018-07-01 22:15:14,2018-07-03 13:57:00,NaT,2018-07-30
82868,0d3268bad9b086af767785e3f0fc0133,4f1d63d35fb7c8999853b2699f5c7649,delivered,2018-07-01 21:14:02,2018-07-01 21:29:54,2018-07-03 09:28:00,NaT,2018-07-24
92643,2d858f451373b04fb5c984a1cc2defaf,e08caf668d499a6d643dafd7c5cc498a,delivered,2017-05-25 23:22:43,2017-05-25 23:30:16,NaT,NaT,2017-06-23
97647,ab7c89dc1bf4a1ead9d6ec1ec8968a84,dd1b84a7286eb4524d52af4256c0ba24,delivered,2018-06-08 12:09:39,2018-06-08 12:36:39,2018-06-12 14:10:00,NaT,2018-06-26
98038,20edc82cf5400ce95e1afacc25798b31,28c37425f1127d887d7337f284080a0f,delivered,2018-06-27 16:09:12,2018-06-27 16:29:30,2018-07-03 19:26:00,NaT,2018-07-19


In [6]:
delivered = orders[orders["order_status"] == "delivered"].dropna(
    subset=["order_delivered_customer_date"]).copy()

delivered["delivery_days"] = (
    delivered["order_delivered_customer_date"] - delivered["order_purchase_timestamp"]
).dt.days
delivered["delay_days"] = (
    delivered["order_delivered_customer_date"] - delivered["order_estimated_delivery_date"]
).dt.days
delivered["is_late"] = delivered["delay_days"] > 0

print(delivered[["delivery_days", "delay_days"]].describe())
print("Share of late orders:", delivered["is_late"].mean())

       delivery_days    delay_days
count   96470.000000  96470.000000
mean       12.093604    -11.875889
std         9.551380     10.182105
min         0.000000   -147.000000
25%         6.000000    -17.000000
50%        10.000000    -12.000000
75%        15.000000     -7.000000
max       209.000000    188.000000
Share of late orders: 0.06773090079817559


In [7]:
products = products.rename(columns={
    "product_name_lenght": "product_name_length",
    "product_description_lenght": "product_description_length",
})
products["product_category_name"] = products["product_category_name"].fillna("unknown")

products = products.merge(category_translation, on="product_category_name", how="left")
products["product_category_name_english"] = products["product_category_name_english"].fillna(
    products["product_category_name"])

products[["product_category_name", "product_category_name_english"]].head()

,product_category_name,product_category_name_english
0,perfumaria,perfumery
1,artes,art
2,esporte_lazer,sports_leisure
3,bebes,baby
4,utilidades_domesticas,housewares


In [8]:
print("Orders with more than one review:", reviews["order_id"].duplicated().sum())

reviews = reviews.sort_values("review_answer_timestamp").drop_duplicates(
    subset="order_id", keep="last")
print(reviews.shape)

Orders with more than one review: 551
(98673, 7)


In [9]:
delivered["is_late"] = delivered["is_late"].astype(int)

orders = orders.merge(
    delivered[["order_id", "delivery_days", "delay_days", "is_late"]],
    on="order_id", how="left"
)
print(orders.shape)   # should still be (99441, 11)

(99441, 11)


In [10]:
missing_cat = products[products["product_category_name_english"] == products["product_category_name"]]
print(missing_cat["product_category_name"].value_counts())

product_category_name
cool_stuff                                       789
pet_shop                                         719
unknown                                          610
consoles_games                                   317
market_place                                     104
audio                                             58
dvds_blu_ray                                      48
la_cuisine                                        10
portateis_cozinha_e_preparadores_de_alimentos     10
pc_gamer                                           3
Name: count, dtype: int64


In [11]:
import sqlite3

conn = sqlite3.connect("../data/olist.db")

clean_tables = {
    "orders": orders, "customers": customers, "items": items,
    "payments": payments, "reviews": reviews, "products": products,
    "sellers": sellers,
}
for name, df in clean_tables.items():
    df.to_sql(name, conn, if_exists="replace", index=False)

print(pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn))

        name
0     orders
1  customers
2      items
3   payments
4    reviews
5   products
6    sellers


In [12]:
query = """
SELECT p.product_category_name_english AS category,
       ROUND(SUM(i.price), 2)          AS revenue,
       COUNT(DISTINCT i.order_id)      AS orders
FROM items i
JOIN orders o   ON o.order_id = i.order_id
JOIN products p ON p.product_id = i.product_id
WHERE o.order_status = 'delivered'
GROUP BY category
ORDER BY revenue DESC
LIMIT 10;
"""
pd.read_sql(query, conn)

,category,revenue,orders
0,health_beauty,1233131.72,8647
1,watches_gifts,1166176.98,5495
2,bed_bath_table,1023434.76,9272
3,sports_leisure,954852.55,7530
4,computers_accessories,888724.61,6530
5,furniture_decor,711927.69,6307
6,housewares,615628.69,5743
7,cool_stuff,610204.10,3559
8,auto,578966.65,3810
9,toys,471286.48,3804


In [13]:
query = """
SELECT strftime('%Y-%m', o.order_purchase_timestamp) AS month,
       ROUND(SUM(i.price), 2)     AS revenue,
       COUNT(DISTINCT o.order_id) AS orders
FROM orders o
JOIN items i ON i.order_id = o.order_id
WHERE o.order_status = 'delivered'
GROUP BY month
ORDER BY month;
"""
pd.read_sql(query, conn)

,month,revenue,orders
0,2016-09,134.97,1
1,2016-10,40325.11,265
2,2016-12,10.90,1
3,2017-01,111798.36,750
4,2017-02,234223.40,1653
5,2017-03,359198.85,2546
6,2017-04,340669.68,2303
7,2017-05,489338.25,3546
8,2017-06,421923.37,3135
9,2017-07,481604.52,3872


In [14]:
query = """
SELECT CASE WHEN o.is_late = 1 THEN 'Late' ELSE 'On time' END AS delivery,
       COUNT(*)                         AS orders,
       ROUND(AVG(r.review_score), 2)    AS avg_review_score
FROM orders o
JOIN reviews r ON r.order_id = o.order_id
WHERE o.order_status = 'delivered'
  AND o.is_late IS NOT NULL
GROUP BY delivery;
"""
pd.read_sql(query, conn)

,delivery,orders,avg_review_score
0,Late,6381,2.27
1,On time,89443,4.29


In [15]:
query = """
WITH cust AS (
    SELECT c.customer_unique_id,
           COUNT(DISTINCT o.order_id) AS n_orders
    FROM orders o
    JOIN customers c ON c.customer_id = o.customer_id
    WHERE o.order_status = 'delivered'
    GROUP BY c.customer_unique_id
)
SELECT CASE WHEN n_orders = 1 THEN 'One-time' ELSE 'Repeat' END AS customer_type,
       COUNT(*) AS customers,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM cust), 2) AS pct
FROM cust
GROUP BY customer_type;
"""
pd.read_sql(query, conn)

,customer_type,customers,pct
0,One-time,90557,97.0
1,Repeat,2801,3.0


In [16]:
query = """
WITH monthly AS (
    SELECT strftime('%Y-%m', o.order_purchase_timestamp) AS month,
           SUM(i.price) AS revenue
    FROM orders o
    JOIN items i ON i.order_id = o.order_id
    WHERE o.order_status = 'delivered'
    GROUP BY month
)
SELECT month,
       ROUND(revenue, 2) AS revenue,
       ROUND(100.0 * (revenue - LAG(revenue) OVER (ORDER BY month))
             / LAG(revenue) OVER (ORDER BY month), 1) AS mom_growth_pct
FROM monthly
WHERE month BETWEEN '2017-01' AND '2018-08'
ORDER BY month;
"""
pd.read_sql(query, conn)

,month,revenue,mom_growth_pct
0,2017-01,111798.36,NaN
1,2017-02,234223.40,109.5
2,2017-03,359198.85,53.4
3,2017-04,340669.68,-5.2
4,2017-05,489338.25,43.6
5,2017-06,421923.37,-13.8
6,2017-07,481604.52,14.1
7,2017-08,554699.70,15.2
8,2017-09,607399.67,9.5
9,2017-10,648247.65,6.7


In [17]:
query = """
SELECT c.customer_state,
       COUNT(*)                          AS orders,
       ROUND(100.0 * AVG(o.is_late), 1)  AS late_pct,
       ROUND(AVG(o.delivery_days), 1)    AS avg_delivery_days
FROM orders o
JOIN customers c ON c.customer_id = o.customer_id
WHERE o.order_status = 'delivered' AND o.is_late IS NOT NULL
GROUP BY c.customer_state
HAVING COUNT(*) >= 100
ORDER BY late_pct DESC;
"""
pd.read_sql(query, conn)

,customer_state,orders,late_pct,avg_delivery_days
0,AL,397,21.4,24.0
1,MA,717,17.4,21.1
2,SE,335,15.2,21.0
3,PI,476,13.9,19.0
4,CE,1279,13.8,20.8
5,BA,3256,12.2,18.9
6,RJ,12350,12.1,14.8
7,PA,946,11.2,23.3
8,ES,1995,10.7,15.3
9,PB,517,10.4,20.0
